In [3]:
import numpy as np
import os
import re

# ================= 設定區 =================
# 主資料夾路徑 (包含所有 tile 資料夾的目錄)
BASE_DIR = "output_data_packaged_COE"
# 最終輸出的單一檔案名稱與路徑
OUTPUT_FILE = "4.8.2_all_layers_weight_bias_64b.txt"
# ========================================

def load_any_64bit(filepath):
    """自動偵測格式並讀取資料 (支援標準 TXT 與 Xilinx COE)"""
    if not os.path.exists(filepath):
        return np.array([], dtype=np.uint64)

    try:
        with open(filepath, 'r') as f:
            content = f.read()

        # 處理 COE 格式：尋找 vector 之後的內容
        if "memory_initialization_vector" in content:
            content = content.split("memory_initialization_vector")[-1]
            content = content.split("=")[-1]
            content = content.replace(",", " ").replace(";", " ")

        # 依照空白、換行分割字串
        hex_tokens = content.split()

        # 過濾：只留下 16 進位的字串
        values = []
        for s in hex_tokens:
            s_clean = s.strip()
            if s_clean:
                try:
                    values.append(int(s_clean, 16))
                except ValueError:
                    continue

        return np.array(values, dtype=np.uint64)
    except Exception as e:
        print(f"[讀取錯誤] {os.path.basename(filepath)}: {e}")
        return np.array([], dtype=np.uint64)

def natural_sort_key(s):
    """用於自然排序，確保 tile2 排在 tile10 前面"""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def process_all_layers():
    print(f"=== 開始批次處理所有層資料 ===")
    print(f"掃描目錄: {BASE_DIR}")

    if not os.path.exists(BASE_DIR):
        print("❌ 找不到主路徑，請檢查 Google Drive 掛載狀況。")
        return

    # 1. 取得所有子資料夾
    subdirs = [d for d in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, d))]

    # 先對所有資料夾進行自然排序，作為打底
    subdirs.sort(key=natural_sort_key)

    # 🌟 核心修改區塊：指定優先排序的清單
    PRIORITY_FOLDERS = [
        "tile0_Conv1",
        "tile17_ConvLast_FC",
        "tile1.1_DownSamplingL"
    ]

    ordered_subdirs = []

    # 步驟 A: 將指定的優先資料夾，依照我們自訂的順序抽出來
    for target in PRIORITY_FOLDERS:
        # 尋找是否有完全符合的資料夾名稱
        match = next((d for d in subdirs if d == target), None)
        if match:
            ordered_subdirs.append(match)
            subdirs.remove(match) # 從原本的名單中剔除
            print(f"📌 已將 {match} 強制排定為第 {len(ordered_subdirs)} 層處理！")
        else:
            print(f"⚠️ 警告：找不到指定的優先資料夾 {target}，將略過此項。")

    # 步驟 B: 把剩下已經自然排序好的資料夾，全數接在後面
    ordered_subdirs.extend(subdirs)

    # 用排好的新名單取代原本的名單
    subdirs = ordered_subdirs

    all_final_data = []
    total_w_count = 0
    total_b_count = 0

    # 2. 依序走訪每一個排序好的資料夾
    for folder_name in subdirs:
        layer_dir = os.path.join(BASE_DIR, folder_name)
        files = os.listdir(layer_dir)

        layer_w_count = 0
        layer_b_count = 0

        # 先處理該層的 Weight
        w_files = sorted([f for f in files if f.startswith("Weight") and (f.endswith(".txt") or f.endswith(".coe"))])
        for w in w_files:
            data = load_any_64bit(os.path.join(layer_dir, w))
            if len(data) > 0:
                all_final_data.append(data)
                layer_w_count += len(data)
                total_w_count += len(data)

        # 再處理該層的 Bias
        b_files = sorted([f for f in files if f.startswith("Bias") and (f.endswith(".txt") or f.endswith(".coe"))])
        for b in b_files:
            data = load_any_64bit(os.path.join(layer_dir, b))
            if len(data) > 0:
                all_final_data.append(data)
                layer_b_count += len(data)
                total_b_count += len(data)

        print(f"✅ 已處理 {folder_name:<25} | Weight: {layer_w_count:>5} 筆 | Bias: {layer_b_count:>5} 筆")

    # 3. 輸出最終的合併檔案
    if all_final_data:
        final_array = np.concatenate(all_final_data)

        print("\n⏳ 正在寫入最終檔案，這可能需要幾秒鐘...")
        with open(OUTPUT_FILE, 'w') as f:
            for val in final_array:
                f.write(f"{int(val):016X}\n")

        print("\n" + "="*50)
        print(f"🎉 轉換與合併大成功！")
        print(f"總共處理層數 : {len(subdirs)}")
        print(f"總 Weight 筆數: {total_w_count}")
        print(f"總 Bias   筆數: {total_b_count}")
        print(f"總合併筆數    : {len(final_array)}")
        print(f"📁 檔案已儲存至: {OUTPUT_FILE}")
        print("="*50)
    else:
        print("❌ 沒讀到任何資料，請檢查資料夾內是否有正確的 txt/coe 檔案。")

if __name__ == "__main__":
    process_all_layers()

=== 開始批次處理所有層資料 ===
掃描目錄: output_data_packaged_COE
📌 已將 tile0_Conv1 強制排定為第 1 層處理！
📌 已將 tile17_ConvLast_FC 強制排定為第 2 層處理！
📌 已將 tile1.1_DownSamplingL 強制排定為第 3 層處理！
✅ 已處理 tile0_Conv1               | Weight:   216 筆 | Bias:    24 筆
✅ 已處理 tile17_ConvLast_FC        | Weight: 51840 筆 | Bias:   504 筆
✅ 已處理 tile1.1_DownSamplingL     | Weight:   216 筆 | Bias:    24 筆
✅ 已處理 tile1.2_DownSamplingR     | Weight:   360 筆 | Bias:    48 筆
✅ 已處理 tile2.2_OGShuffle         | Weight:   360 筆 | Bias:    48 筆
✅ 已處理 tile3.2_OGShuffle         | Weight:   360 筆 | Bias:    48 筆
✅ 已處理 tile4.2_OGShuffle         | Weight:   360 筆 | Bias:    48 筆
✅ 已處理 tile5.1_DownSamplingL     | Weight:   720 筆 | Bias:    48 筆
✅ 已處理 tile5.2_DownSamplingR     | Weight:  1296 筆 | Bias:    72 筆
✅ 已處理 tile6.2_OGShuffle         | Weight:  1296 筆 | Bias:    72 筆
✅ 已處理 tile7.2_OGShuffle         | Weight:  1296 筆 | Bias:    72 筆
✅ 已處理 tile8.2_OGShuffle         | Weight:  1296 筆 | Bias:    72 筆
✅ 已處理 tile9.2_OGShuffle         | Weight:  1296

In [4]:
import numpy as np

# 設定路徑
input_txt = "4.8.2_all_layers_weight_bias_64b.txt"
output_bin = "4.8.2_all_layers_weight_bias_64b.bin"

print(f"正在轉換 {input_txt} ...")

# 讀取文字檔中的 16 進位字串並轉為 uint64
data = np.genfromtxt(input_txt, dtype=str)
hex_values = np.array([int(x, 16) for x in data], dtype=np.uint64)

# 寫入二進位檔
hex_values.tofile(output_bin)

print(f"✅ 轉換完成！產出檔案：{output_bin}")
print(f"檔案大小: {len(hex_values) * 8} bytes")

正在轉換 4.8.2_all_layers_weight_bias_64b.txt ...
✅ 轉換完成！產出檔案：4.8.2_all_layers_weight_bias_64b.bin
檔案大小: 712128 bytes
